# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [ ]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

In [ ]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [11]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# Load the dataset here so this task works when run on its own.
df = px.data.gapminder()

continent = "Europe"
years = [2002, 2007]
task1_df = df.loc[
    (df["continent"] == continent) & df["year"].isin(years)
].copy()

# Draw each country as a thin grey path first.
task1_fig = go.Figure()
for _, country_data in task1_df.groupby("country"):
    country_data = country_data.sort_values("year")
    task1_fig.add_trace(
        go.Scatter(
            x=country_data["gdpPercap"],
            y=country_data["lifeExp"],
            mode="lines",
            line={"color": "rgba(120, 120, 120, 0.28)", "width": 1},
            showlegend=False,
            hoverinfo="skip",
        )
    )

# Overlay the two highlighted years as points.
points = px.scatter(
    task1_df,
    x="gdpPercap",
    y="lifeExp",
    color=task1_df["year"].astype(str),
    color_discrete_map={"2002": "#636EFA", "2007": "#EF553B"},
    hover_name="country",
    log_x=True,
    labels={
        "gdpPercap": "GDP per capita (US$, log scale)",
        "lifeExp": "Life expectancy (years)",
        "color": "Year",
    },
)
for trace in points.data:
    task1_fig.add_trace(trace)

# Add an annotation for a clear example.
germany = task1_df.loc[task1_df["country"] == "Germany"].sort_values("year")
if len(germany) == 2:
    task1_fig.add_annotation(
        x=germany.iloc[1]["gdpPercap"],
        y=germany.iloc[1]["lifeExp"],
        ax=germany.iloc[0]["gdpPercap"],
        ay=germany.iloc[0]["lifeExp"],
        xref="x",
        yref="y",
        axref="x",
        ayref="y",
        text="Germany: 2002 → 2007",
        showarrow=True,
        arrowhead=2,
        bgcolor="white",
    )

task1_fig.update_xaxes(type="log", title_text="GDP per capita (US$, log scale)")
task1_fig.update_yaxes(title_text="Life expectancy (years)")
task1_fig.update_layout(
    title=f"{continent}: countries generally gained income and longevity (2002–2007)",
    legend_title_text="Year",
    template="plotly_white",
)
task1_fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [12]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# Task 2: all countries in 2007, highlight one continent.
df = px.data.gapminder()
all_2007 = df[df["year"] == 2007].copy()
highlight_continent = "Europe"

# Use a grey base layer for all countries.
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=all_2007["gdpPercap"],
        y=all_2007["lifeExp"],
        mode="markers",
        marker=dict(
            size=all_2007["pop"] / 1_000_000,
            sizemode="area",
            sizeref=2 * all_2007["pop"].max() / (45 ** 2),
            sizemin=4,
            color="rgba(140, 140, 140, 0.45)",
            line=dict(color="rgba(0, 0, 0, 0.25)", width=0.8),
        ),
        customdata=all_2007[["country", "pop"]].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "GDP: %{x:$,.0f}<br>"
            "Life expectancy: %{y:.1f}<br>"
            "Population: %{customdata[1]:,.0f}<extra></extra>"
        ),
        showlegend=False,
        name="All countries",
    )
)

highlight_df = all_2007[all_2007["continent"] == highlight_continent].copy()
fig.add_trace(
    go.Scatter(
        x=highlight_df["gdpPercap"],
        y=highlight_df["lifeExp"],
        mode="markers",
        marker=dict(
            size=highlight_df["pop"] / 1_000_000,
            sizemode="area",
            sizeref=2 * all_2007["pop"].max() / (45 ** 2),
            sizemin=4,
            color="#E15759",
            line=dict(color="rgba(0, 0, 0, 0.7)", width=1.1),
        ),
        customdata=highlight_df[["country", "pop"]].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "GDP: %{x:$,.0f}<br>"
            "Life expectancy: %{y:.1f}<br>"
            "Population: %{customdata[1]:,.0f}<extra></extra>"
        ),
        name=highlight_continent,
    )
)

fig.add_annotation(
    x=highlight_df["gdpPercap"].max(),
    y=highlight_df["lifeExp"].max(),
    text="Europe combines high income with longer life expectancy",
    showarrow=True,
    arrowhead=2,
    ax=highlight_df["gdpPercap"].median(),
    ay=highlight_df["lifeExp"].median() + 6,
    bgcolor="white",
    bordercolor="rgba(0, 0, 0, 0.15)",
)

fig.update_xaxes(type="log", title_text="GDP per capita (US$, log scale)")
fig.update_yaxes(title_text="Life expectancy (years)")
fig.update_layout(
    title=f"{highlight_continent} stands out in 2007: higher income and longer lives",
    template="plotly_white",
    legend_title_text="Continent",
    margin=dict(l=60, r=20, t=60, b=60),
)
fig.show()
